# Autonomous Systems Portfolio 2  
***StarGunner met Reinforced Learning***

|Name     |Studentnumber|Github    |
|---------|-------------|----------|
|Henry Lau|22122958     |HenryLau08|
|Michal Reszka-Gniecki|23025069|ckires|
|Mohamed Belaachir|22143572|mobelaachir|

In [1]:
import gymnasium as gym
import ale_py
import math
import numpy as np
import random
import matplotlib
import matplotlib.pyplot as plt
from collections import namedtuple, deque
from itertools import count

import tensorflow as tf

gym.register_envs(ale_py)

In [2]:
# set up matplotlib
is_ipython = 'inline' in matplotlib.get_backend()
if is_ipython:
    from IPython import display

plt.ion()

In [3]:
env_id = 'StarGunner-v4'
env = gym.make(env_id, obs_type="ram", full_action_space=True, mode=0)

In [4]:
observation_space = env.observation_space
state_size = observation_space.shape[0]
print(f'State space: {state_size}')

action_size = env.action_space.n
print(f'Action space: {action_size}')

possible_actions = list(range(0,int(action_size)))

State space: 128
Action space: 18


In [5]:
def preprocess_ram(state):
    state = np.array(state, dtype=np.float32)
    # state = state / 255.0 # Normalize
    return tf.convert_to_tensor(state, dtype=tf.float32)

In [6]:
class DQN(tf.keras.Model):
    def __init__(self, action_size):
        super().__init__()

        self.fc1 = tf.keras.layers.Dense(128, activation="relu")
        self.fc2 = tf.keras.layers.Dense(128, activation="relu")
        self.out = tf.keras.layers.Dense(action_size)

    def call(self, x):
        x = tf.cast(x, tf.float32)
        x = self.fc1(x)
        x = self.fc2(x)
        return self.out(x)

In [7]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def add(self, s, a, r, s2, d):
        self.buffer.append((
            s.astype(np.uint8),     # was float32 /255 before — undo that
            np.int32(a),
            np.float32(r),
            s2.astype(np.uint8),
            np.bool_(d),
        ))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, s2, d = zip(*batch)
        s = np.array(s, dtype=np.float32)
        a = np.array(a, dtype=np.int32)
        r = np.array(r, dtype=np.float32)
        s2 = np.array(s2, dtype=np.float32)
        d = np.array(d, dtype=np.float32)
        return s, a, r, s2, d

    def __len__(self):
        return len(self.buffer)

In [8]:
def select_action(state, epsilon, model, action_dim):
    """
    Explore  (prob ε)  → random action.
    Exploit (prob 1-ε) → argmax Q(s, a; θ).
    state is already float32 and normalised before being passed in.
    """
    if np.random.rand() < epsilon:
        return np.random.randint(action_dim)        # explore

    state_t  = tf.expand_dims(state, axis=0)      # (1, 128)
    q_values = model(state_t, training=False)      # (1, action_dim)
    return int(tf.argmax(q_values[0]).numpy())      # scalar int


def decay_epsilon(epsilon, eps_end=0.01, decay=0.995):
    return max(eps_end, epsilon * decay)

In [9]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)

@tf.function   # compiles to a graph — faster than eager mode
def train_step(states, actions, rewards, next_states, dones,
               q_net, target_net, gamma=0.99):
    """
    1. Compute Bellman target y with the FROZEN target network.
       (No gradient flows through target_net — that's the point.)

    2. Inside GradientTape: forward pass through q_net, compute MSE.

    3. tape.gradient gives us  ∂L/∂θ  for every weight in q_net.

    4. optimizer.apply_gradients nudges weights to reduce L,
       which pushes Q(s,a) → y → better reward predictions.
    """

    # ── step 1: Bellman target (no gradient) ────────────────────
    next_q     = target_net(next_states, training=False) # (B, action_dim)
    max_next_q = tf.reduce_max(next_q, axis=1)           # (B,)
    # y = r + γ·max Q(s',a';θ⁻)·(1-done)
    y = rewards + gamma * max_next_q * (1.0 - dones)     # (B,)

    # ── step 2: forward pass + loss inside tape ─────────────────
    with tf.GradientTape() as tape:
        q_vals = q_net(states, training=True)            # (B, action_dim)

        # pull out Q(s, a) for the action actually taken
        idx  = tf.stack([tf.range(tf.shape(actions)[0]),
                         actions], axis=1)               # (B, 2) indices
        q_sa = tf.gather_nd(q_vals, idx)                  # (B,)

        # MSE:  L(θ) = mean( (y - Q(s,a;θ))² )
        loss = tf.reduce_mean(tf.square(y - q_sa))

    # ── step 3: ∂L/∂θ ───────────────────────────────────────────
    grads = tape.gradient(loss, q_net.trainable_variables)

    # ── step 4: θ ← θ - α·∂L/∂θ ────────────────────────────────
    optimizer.apply_gradients(zip(grads, q_net.trainable_variables))

    return loss

In [10]:
def compute_loss(batch, q_net, target_net, gamma):
    """
    L(θ) = E[ (y − Q(s,a;θ))² ]

    y = r + γ · max_a' Q(s',a';θ⁻)   if not done
    y = r                              if done
    """

    states, actions, rewards, next_states, dones = batch

    states      = tf.convert_to_tensor(states, dtype=tf.float32)   # (B, state_dim)
    actions     = tf.convert_to_tensor(actions, dtype=tf.int32)    # (B,)
    rewards     = tf.convert_to_tensor(rewards, dtype=tf.float32)   # (B,)
    next_states = tf.convert_to_tensor(next_states, dtype=tf.float32)
    dones       = tf.convert_to_tensor(dones, dtype=tf.float32)

    # Q(s, a; θ)
    q_vals = q_net(states)  # (B, num_actions)

    # gather Q-values for taken actions
    batch_indices = tf.range(tf.shape(actions)[0])
    action_indices = tf.stack([batch_indices, actions], axis=1)
    q_sa = tf.gather_nd(q_vals, action_indices)  # (B,)

    # Q(s', a'; θ⁻) with no gradient
    next_q = target_net(next_states)             # (B, num_actions)
    next_q = tf.reduce_max(next_q, axis=1)       # (B,)

    # Bellman target
    y = rewards + gamma * next_q * (1.0 - dones)

    # MSE loss
    loss = tf.reduce_mean(tf.square(q_sa - y))

    return loss

In [ ]:
def train():
    env_id = 'StarGunner-v4'
    env = gym.make(env_id, obs_type="ram", full_action_space=True)
    state_dim  = env.observation_space.shape[0]   # 128
    action_dim = int(env.action_space.n)          # 18

    # ── networks ─────────────────────────────────────────────────
    q_net      = DQN(action_dim)
    target_net = DQN(action_dim)

    dummy = tf.zeros((1, state_dim))

    q_net(dummy)
    target_net(dummy)

    target_net.set_weights(q_net.get_weights())    # start identical

    buffer = ReplayBuffer(500_000)

    # ── hyperparameters ──────────────────────────────────────────
    GAMMA         = 0.99
    BATCH_SIZE    = 5000
    EPS_START     = 0.99
    EPS_END       = 0.05
    EPS_DECAY     = 0.99995
    TARGET_UPDATE = 100
    MAX_EPISODES  = 200

    epsilon = EPS_START
    step    = 0

    for ep in range(MAX_EPISODES):
        state, _ = env.reset()
        state    = state.astype(np.float32)  # uint8 → [0,1]
        total_r  = 0

        while True:
            # act
            action = select_action(state, epsilon, q_net, action_dim)
            ns, r, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            ns   = ns.astype(np.float32)

            buffer.add(state, action, r, ns, float(done))
            state    = ns
            total_r += r
            step    += 1

            # learn — only once buffer has enough samples
            if len(buffer) >= BATCH_SIZE:
                s, a, r_b, s2, d = buffer.sample(BATCH_SIZE)
                loss = train_step(s, a, r_b, s2, d,
                                  q_net, target_net, GAMMA)

            # hard-copy online → target every TARGET_UPDATE steps
            if step % TARGET_UPDATE == 0:
                target_net.set_weights(q_net.get_weights())

            epsilon = decay_epsilon(epsilon=epsilon, eps_end=EPS_END, decay=EPS_DECAY)

            if done:
                break

        print(f"ep {ep+1:4d} | reward {total_r:7.1f} | ε {epsilon:.3f}")

    env.close()

if __name__ == "__main__":
    train()

ep    1 | reward   600.0 | ε 0.929
ep    2 | reward  1300.0 | ε 0.819
ep    3 | reward   200.0 | ε 0.767
ep    4 | reward   600.0 | ε 0.708
ep    5 | reward   700.0 | ε 0.643
ep    6 | reward  1200.0 | ε 0.590
ep    7 | reward  1000.0 | ε 0.555
ep    8 | reward  1100.0 | ε 0.498
ep    9 | reward  1100.0 | ε 0.468
ep   10 | reward  1200.0 | ε 0.426
ep   11 | reward   400.0 | ε 0.402
ep   12 | reward   800.0 | ε 0.372
ep   13 | reward  1100.0 | ε 0.349
ep   14 | reward  3600.0 | ε 0.300
ep   15 | reward   900.0 | ε 0.283
ep   16 | reward  1600.0 | ε 0.252
ep   17 | reward   800.0 | ε 0.241
ep   18 | reward   800.0 | ε 0.222
ep   19 | reward   200.0 | ε 0.211
ep   20 | reward   500.0 | ε 0.200
ep   21 | reward   800.0 | ε 0.189
ep   22 | reward  1400.0 | ε 0.169
ep   23 | reward   900.0 | ε 0.151
ep   24 | reward   900.0 | ε 0.138
ep   25 | reward   900.0 | ε 0.124
ep   26 | reward   800.0 | ε 0.114
ep   27 | reward   100.0 | ε 0.108
ep   28 | reward   400.0 | ε 0.097
ep   29 | reward   4